# LoMa matcher on CPU (davnords/LoMa)

Runs [LoMa](https://github.com/davnords/LoMa) ([arXiv:2604.04931](https://arxiv.org/abs/2604.04931)) and the pre-registered CPU clear-experiment from [`apps/loma-matcher`](https://github.com/Ryukijano/tech-demos/tree/main/apps/loma-matcher).

* **Runs on the free CPU runtime.** Expect about 8–12 s per `match()` on a few CPU cores (Colab's 2-core CPU will be slower). A GPU is optional. It is only used if you switch the runtime yourself; on the free tier a GPU may be unavailable, and paid runtimes cost your own budget.
* Weights (LoMa-B / LoMa-R / DaD / DINOv2) auto-download at run time. No HF token.
* Upstream is MIT; the matcher inherits Apache-2.0 from LightGlue.

In [ ]:
!pip -q install torch torchvision --index-url https://download.pytorch.org/whl/cpu
!pip -q install numpy opencv-python-headless Pillow pyyaml pandas matplotlib einops tqdm tyro
!pip -q install "lomatch @ git+https://github.com/davnords/LoMa.git"
!git clone -q --depth 1 --filter=blob:none --sparse https://github.com/Ryukijano/tech-demos.git
%cd tech-demos
!git sparse-checkout set apps/loma-matcher
%cd apps/loma-matcher

## 1. Match one upstream pair with LoMa-B

In [ ]:
import sys, cv2, matplotlib.pyplot as plt
sys.path.insert(0, 'src')
import lmx
a = lmx.fetch('github:davnords/LoMa/assets/0015_A.jpg')
b = lmx.fetch('github:davnords/LoMa/assets/0015_B.jpg')
model, load_s = lmx.load_model('LoMa-B')
print(f'loaded in {load_s:.1f}s')
ka, kb, t = lmx.match_timed(model, a, b)
nin, ratio = lmx.magsac_inliers(ka, kb, dict(ransacReprojThreshold=0.2, method='USAC_MAGSAC', confidence=0.999999, maxIters=10000))
print(f'match {t:.1f}s  n={len(ka)}  MAGSAC inliers={nin}  ratio={ratio:.3f}')
canvas = lmx.draw_matches(a, b, ka, kb)
plt.figure(figsize=(14, 6)); plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)); plt.axis('off')

## 2. Full pre-registered experiment (seeds 0/1/2 + rotation breakpoints)
Thresholds are fixed in `experiments/exp1_cpu_match/config.yaml`. A full CPU run is about 45 match passes and may take 15–30 min on Colab CPU. Add `--skip-breakpoints` for a quicker run. GPU is optional and not required.

In [ ]:
del model
!python src/run_experiment.py
!python src/analyze.py
!cat experiments/exp1_cpu_match/results.md

In [ ]:
from IPython.display import Image, display
display(Image('experiments/exp1_cpu_match/plot.png'))
!cat experiments/exp1_cpu_match/breakpoints.md